# ResNet18 — Brain Tumour MRI Classification

**Owner:** Member 2 — Suryansh Singh Rathore (230953558)  
**Project:** A Comparative Study of Deep Learning Architectures for Brain Tumour MRI Classification (ICT-4442)

Residual CNN: ImageNet-pretrained ResNet18 with a new 4-way classifier, fully fine-tuned.

All four models share the same data split, preprocessing and training/evaluation code (`common/`); only the architecture and learning rate differ.

> The printed outputs below are from the original run on a Kaggle GPU. The last cell writes the files the comparison scripts in `analysis/` read.

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root, so `common` and `models` can be imported

import torch
from common.config import CLASSES, OUTPUT_DIR, device, set_seed
from common.data import describe_dataset, get_loaders
from common.engine import train_model, evaluate_model, get_probs_and_labels
from common.results import save_run

print(f"Device: {device}")
set_seed()

Device: cuda


## 2. Data (shared split)

In [2]:
describe_dataset()

Train classes found: ['glioma', 'meningioma', 'notumor', 'pituitary']
Test classes found:  ['glioma', 'meningioma', 'notumor', 'pituitary']
  glioma: 1400 images
  meningioma: 1400 images
  notumor: 1400 images
  pituitary: 1400 images


In [3]:
train_loader, val_loader, test_loader = get_loaders()

Train: 5040 | Val: 560 | Test: 1600


## 3. Model — ResNet18

In [4]:
from models.resnet18 import get_resnet18

n_params = sum(p.numel() for p in get_resnet18().parameters())
print(f"ResNet18 parameters: {n_params:,}")

ResNet18 parameters: 11,178,564


## 4. Train (10 epochs, Adam, cross-entropy) and evaluate on the test set

In [5]:
resnet_model = get_resnet18(num_classes=len(CLASSES))
resnet_model, resnet_history = train_model(resnet_model, train_loader, val_loader, epochs=10, model_name="ResNet18")

resnet_loss, resnet_acc, resnet_prec, resnet_rec, resnet_f1, resnet_cm = evaluate_model(resnet_model, test_loader)
print(f"\nResNet18 Test Results — Acc: {resnet_acc:.4f}  Precision: {resnet_prec:.4f}  Recall: {resnet_rec:.4f}  F1: {resnet_f1:.4f}")

torch.save(resnet_model.state_dict(), OUTPUT_DIR / 'resnet18_final.pth')

[ResNet18] Epoch 1/10  Train Loss: 0.2611  Train Acc: 0.8986  Val Loss: 0.1411  Val Acc: 0.9500
[ResNet18] Epoch 2/10  Train Loss: 0.0926  Train Acc: 0.9692  Val Loss: 0.0730  Val Acc: 0.9768
[ResNet18] Epoch 3/10  Train Loss: 0.0526  Train Acc: 0.9823  Val Loss: 0.0859  Val Acc: 0.9696
[ResNet18] Epoch 4/10  Train Loss: 0.0369  Train Acc: 0.9877  Val Loss: 0.0912  Val Acc: 0.9696
[ResNet18] Epoch 5/10  Train Loss: 0.0261  Train Acc: 0.9911  Val Loss: 0.0880  Val Acc: 0.9768
[ResNet18] Epoch 6/10  Train Loss: 0.0206  Train Acc: 0.9935  Val Loss: 0.0534  Val Acc: 0.9839
[ResNet18] Epoch 7/10  Train Loss: 0.0215  Train Acc: 0.9923  Val Loss: 0.0666  Val Acc: 0.9804
[ResNet18] Epoch 8/10  Train Loss: 0.0264  Train Acc: 0.9915  Val Loss: 0.0620  Val Acc: 0.9839
[ResNet18] Epoch 9/10  Train Loss: 0.0202  Train Acc: 0.9942  Val Loss: 0.0353  Val Acc: 0.9893
[ResNet18] Epoch 10/10  Train Loss: 0.0161  Train Acc: 0.9958  Val Loss: 0.0725  Val Acc: 0.9750

ResNet18 Test Results — Acc: 0.9406  P

## 5. Save results for the comparison scripts

In [6]:
save_run("resnet18", resnet_model, resnet_history, test_loader)